# 04 – CARMA Initialisation from ARMA Parameters

Converts the discrete ARMA parameters (from notebook 03) to CARMA starting values  
for the MLE optimisation in notebook 05.

**This notebook is a helper only.** It does NOT produce the final CARMA parameters —  
that is done in notebook 05 via maximum likelihood.

### Key fixes vs. original
- **CE-4 fixed**: complex ARMA roots are handled correctly via `np.poly`, which  
  recovers real AR coefficients from conjugate-pair eigenvalues without discarding  
  imaginary parts.
- **CE-2 fixed**: the fragile quadratic formula for b0 is removed; b0 is initialised  
  at a simple heuristic and refined by MLE.
- **LI-3 fixed**: eigenvalues are converted to **year⁻¹ units** (multiply per-hour  
  values by 8760), consistent with the paper's calendar-time convention.

In [1]:
import sys, os, json
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

import numpy as np
import warnings

from config import RES_DIR, HOURS_PER_YEAR
from carma_utils import build_companion, arma_to_carma_init

# Load ARMA selections from notebook 03
with open(RES_DIR / 'arma_selected.json') as f:
    arma = json.load(f)

print('ARMA selections loaded:')
for key, val in arma.items():
    print(f'  {key}: p={val["p"]}, q={val["q"]}, sigma2={val["sigma2"]:.4f}')

ARMA selections loaded:
  temperature: p=4, q=3, sigma2=0.3759
  logprice: p=4, q=2, sigma2=0.0006


## 1.  Temperature CARMA initialisation

In [2]:
t_pars = arma['temperature']
p_t, q_t = t_pars['p'], t_pars['q']

theta0_list_temp = arma_to_carma_init(
    phi_ar       = t_pars['phi'],
    sigma2_arma  = t_pars['sigma2'],
    p            = p_t,
    q            = q_t,
    hours_per_year = HOURS_PER_YEAR,
    n_random     = 5,
)

print(f'Temperature CARMA({p_t},{q_t}) – starting points (p+q+1 = {p_t+q_t+1} params each):')
for i, th in enumerate(theta0_list_temp):
    a_c = th[:p_t];  b_c = np.append(th[p_t:p_t+q_t], 1.0);  sigma_c = np.exp(th[p_t+q_t])
    A = build_companion(a_c)
    eigs = np.linalg.eigvals(A)
    stable = 'OK' if np.all(eigs.real < 0) else 'UNSTABLE'
    print(f'  start {i+1}: a={np.round(a_c,3)}  b={np.round(b_c,3)}  sigma={sigma_c:.3f}  [{stable}]')

Temperature CARMA(4,3) – starting points (p+q+1 = 8 params each):
  start 1: a=[1.02900250e+04 9.21979041e+07 5.24756174e+11 1.25183190e+14]  b=[1. 1. 1. 1.]  sigma=28.690  [OK]
  start 2: a=[1.02900630e+04 9.21979041e+07 5.24756174e+11 1.25183190e+14]  b=[0.939 1.208 1.491 1.   ]  sigma=38.118  [OK]
  start 3: a=[1.02898140e+04 9.21979038e+07 5.24756174e+11 1.25183190e+14]  b=[0.402 1.034 0.726 1.   ]  sigma=23.032  [OK]
  start 4: a=[1.02898620e+04 9.21979040e+07 5.24756174e+11 1.25183190e+14]  b=[1.061 1.51  0.9   1.   ]  sigma=31.881  [OK]
  start 5: a=[1.02902960e+04 9.21979042e+07 5.24756174e+11 1.25183190e+14]  b=[0.963 1.166 0.797 1.   ]  sigma=26.945  [OK]


## 2.  Log-price CARMA initialisation

In [3]:
p_pars = arma['logprice']
p_p, q_p = p_pars['p'], p_pars['q']

theta0_list_price = arma_to_carma_init(
    phi_ar       = p_pars['phi'],
    sigma2_arma  = p_pars['sigma2'],
    p            = p_p,
    q            = q_p,
    hours_per_year = HOURS_PER_YEAR,
    n_random     = 5,
)

print(f'Log-price CARMA({p_p},{q_p}) – starting points:')
for i, th in enumerate(theta0_list_price):
    a_c = th[:p_p];  b_c = np.append(th[p_p:p_p+q_p], 1.0);  sigma_c = np.exp(th[p_p+q_p])
    A = build_companion(a_c)
    eigs = np.linalg.eigvals(A)
    stable = 'OK' if np.all(eigs.real < 0) else 'UNSTABLE'
    print(f'  start {i+1}: a={np.round(a_c,3)}  b={np.round(b_c,3)}  sigma={sigma_c:.3f}  [{stable}]')

Log-price CARMA(4,2) – starting points:
  start 1: a=[1.42951560e+04 1.33934087e+08 3.91305872e+11 1.64712652e+14]  b=[1. 1. 1.]  sigma=1.309  [OK]
  start 2: a=[1.42951930e+04 1.33934087e+08 3.91305872e+11 1.64712652e+14]  b=[0.939 1.208 1.   ]  sigma=1.936  [OK]
  start 3: a=[1.42954400e+04 1.33934087e+08 3.91305872e+11 1.64712652e+14]  b=[1.112 0.402 1.   ]  sigma=1.226  [OK]
  start 4: a=[1.42947820e+04 1.33934087e+08 3.91305872e+11 1.64712652e+14]  b=[1.223 1.413 1.   ]  sigma=1.260  [OK]
  start 5: a=[1.42955660e+04 1.33934087e+08 3.91305872e+11 1.64712652e+14]  b=[1.128 0.877 1.   ]  sigma=0.993  [OK]


## 3.  Verify the eigenvalue conversion

The CARMA eigenvalues should have negative real parts (stability).  
They are in **year⁻¹ units** — characteristic time = 1/|λ| years.

In [4]:
for label, pars, theta0_list in [
        ('Temperature', t_pars, theta0_list_temp),
        ('Log-price',   p_pars, theta0_list_price),
    ]:
    print(f'\n{label}:')
    p = pars['p']; q = pars['q']
    th = theta0_list[0]   # ARMA-derived start (no perturbation)
    a_c = th[:p]
    A = build_companion(a_c)
    eigs = np.linalg.eigvals(A)
    for ev in eigs:
        tau_years = -1.0 / ev.real
        tau_hours = tau_years * HOURS_PER_YEAR
        print(f'  eigenvalue = {ev:.4f}  →  τ = {tau_years:.4f} yr = {tau_hours:.1f} h')


Temperature:
  eigenvalue = -1401.3070+8212.5751j  →  τ = 0.0007 yr = 6.3 h
  eigenvalue = -1401.3070-8212.5751j  →  τ = 0.0007 yr = 6.3 h
  eigenvalue = -7238.2440+0.0000j  →  τ = 0.0001 yr = 1.2 h
  eigenvalue = -249.1669+0.0000j  →  τ = 0.0040 yr = 35.2 h

Log-price:
  eigenvalue = -5078.5674+8017.6797j  →  τ = 0.0002 yr = 1.7 h
  eigenvalue = -5078.5674-8017.6797j  →  τ = 0.0002 yr = 1.7 h
  eigenvalue = -3634.9568+0.0000j  →  τ = 0.0003 yr = 2.4 h
  eigenvalue = -503.0640+0.0000j  →  τ = 0.0020 yr = 17.4 h


## 4.  Save starting points

In [5]:
init_data = {
    'temperature': {
        'p': p_t, 'q': q_t,
        'theta0_list': [th.tolist() for th in theta0_list_temp]
    },
    'logprice': {
        'p': p_p, 'q': q_p,
        'theta0_list': [th.tolist() for th in theta0_list_price]
    },
}

out_path = RES_DIR / 'carma_init.json'
with open(out_path, 'w') as f:
    json.dump(init_data, f, indent=2)
print(f'Saved → {out_path}')

Saved → /home/sven/Nextcloud/Research/Projects/Open/GK25:CARMA-QUANTO/Pricing_and_Hedging_of_Quanto_Options_in_Renewable_Energy_Markets_using_CARMA/Code/notebooks/results/carma_init.json
